In [1]:
from platform import python_version
print(python_version())

3.11.14


### Open Targets

- disease 
- drug_mechanism_of_action 
- evidence_cancer_gene_census 
- evidence_reactome 
- interactions 
- pharmacogenomics 
- study

https://www.opentargets.org/

find a gene:
https://platform.opentargets.org/target/ENSG00000153395/associations

download:
https://platform.opentargets.org/downloads

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)


from libs.Basic import pdwritecsv, pdreadcsv, create_dir, write_txt
from libs.enricher_lib import enricheR
from libs.config_lib import Config
from libs.open_target_lib import OpenTarget

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root_ot { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
               root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
               case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
               std_filename=std_filename, std_filename_list=std_filename_list,
               geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
               tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
               LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
               num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
               min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
               saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", enr.root0, enr.root_disease)
case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(enr.echo_parameters())


Start opening tables ....
Building synonym dictionary ...

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC
>>> Tumor
>>> case Tumor
	DEGs 20006
		Up (#10358)
		Dw (#9648)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    12
1                      IG_C_pseudogene     4
2                            IG_D_gene     1
3                            IG_J_gene     9
4                      IG_J_pseudogene     1
5                            IG_V_gene   124
6                      IG_V_pseudogene    50
7                              Mt_tRNA    17
8                                  TEC   112
9                            TR_C_gene     5
10                           TR_D_gene     1
11                           TR_J_gene    10
12                           TR_V_gene    69
13                     TR_V_pseudogene     1
14                              lncRNA  3193
15                               miRNA   

In [5]:
enr.root_kegg, enr.kegg_fname

(PosixPath('/home/flavio/uv/perturb_agent/data/colab/kegg'),
 'kegg_pathways.tsv')

In [6]:
enr.enr_db_list

['Reactome_Pathways_2024']

In [7]:
print(len(enr.gene.df_my_gene))
enr.gene.df_my_gene.head(2)

27223


,entrezid,symbol,geneid,name,synonyms,other_names,ec_enzyme,refseq_gen,refseq_prot,refseq_rna,...,acessions_rna,acessions_trans,map_location,dic_panther,ortholog,dic_uniprot,swissprot,wikipedia,refseq_gen,refseq_rna
0,ENSG00000121410,A1BG,1,alpha-1-B glycoprotein,"['A1B', 'ABG', 'GAB', 'HYST2477']","['HEL-S-163pA', 'alpha-1B-glycoprotein', 'epididymis secretory sperm binding...",NaN,NaN,NP_570602.2,NaN,...,"['AA484435.1', 'AB073611.1', 'AF414429.1', 'AI022193.1', 'AK055885.1', 'AK05...","[{'protein': 'AAH35719.1', 'rna': 'BC035719.1'}, {'protein': 'BAG51591.1', '...",19q13.43,"{'HGNC': '5', '_license': 'http://pantherdb.org/tou.jsp', 'ortholog': [{'MGI...","[{'MGI': '2152878', 'ortholog_type': 'LDO', 'panther_family': 'PTHR11738', '...","{'Swiss-Prot': 'P04217', 'TrEMBL': ['M0R009', 'V9HWD8']}",P04217,{'url_stub': 'A1BG (gene)'},"['NC_000019.10', 'NC_060943.1']",NM_130786.4
1,ENSG00000268895,A1BG-AS1,503538,A1BG antisense RNA 1,"['A1BG-AS', 'A1BGAS', 'NCRNA00181']","['A1BG antisense RNA (non-protein coding)', 'A1BG antisense RNA 1 (non-prote...",NaN,NaN,NaN,NaN,...,"['AK027222.1', 'BC006144.1', 'BC040926.1', 'NR_015380.2']",NaN,19q13.43,NaN,NaN,NaN,NaN,NaN,"['NC_000019.10', 'NC_060943.1']",NR_015380.2


### Keneddy Pathway

https://www.wikipathways.org/pathways/WP3933.html

![Kennedy Pathway](../figures/WP3933_kennety_pathway.svg)

### Open Targets

```Bash
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/evidence_cancer_gene_census .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/drug_mechanism_of_action .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/known_drug .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/pharmacogenomics .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/study .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/evidence_reactome .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/association_by_datasource_direct .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/association_overall_direct .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/target .
colab/
    open_targets/
    ├── target/
    ├── disease/
    ├── drug_mechanism_of_action/
    ├── known_drug
    ├── evidence_cancer_gene_census/
    ├── evidence_reactome/
    ├── interactions/
    ├── pharmacogenomics/
    ├── association_by_datasource_direct/
    ├── association_overall_direct/
    └── study/
```

In [8]:
ot = OpenTarget(root_colab = root_colab)

### Is target ok?

In [9]:
ot.is_installed()

root exists: True
root is dir: True

Top-level content:
DIR  association_by_datasource_direct
DIR  association_overall_direct
DIR  disease
DIR  drug_mechanism_of_action
DIR  evidence_cancer_gene_census
DIR  evidence_reactome
FILE index.html
DIR  interactions
DIR  known_drug
DIR  pharmacogenomics
FILE robots.txt
DIR  study
DIR  target


In [10]:
ot.resolve_target("TP53")

,target_id,symbol,name,biotype
0,ENSG00000141510,TP53,tumor protein p53,protein_coding


### Testing

In [14]:
symbolA = "FOS"
symbolB = "JUN"
symbolC = None

target_idA, target_idB, df = ot.has_target_interactions(gene_or_ensemblA=symbolA, gene_or_ensemblB=symbolB)

print(len(df), f' interactions between {symbolA} ({target_idA}) and {symbolB} ({target_idB}).' )
df

5  interactions between FOS (ENSG00000170345) and JUN (ENSG00000177606).


,sourceDatabase,targetA,intA,intABiologicalRole,targetB,intB,intBBiologicalRole,speciesA,speciesB,count,scoring
0,intact,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,competitor,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",1,0.980
1,intact,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,unspecified role,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",44,0.980
2,reactome,ENSG00000170345,P01100,unspecified role,ENSG00000177606,P05412,unspecified role,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",3,NaN
3,signor,ENSG00000170345,P01100,regulator,ENSG00000177606,P05412,regulator target,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",1,NaN
4,string,ENSG00000170345,ENSP00000306245,unspecified role,ENSG00000177606,ENSP00000360266,unspecified role,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}",4,0.999


In [ ]:
symbol = 'LPCAT1'
symbol = 'PTGS2'
symbol = 'COX2'
symbol = 'CD274'
symbol = 'PTEN'
symbol = 'KLF5'
symbol = 'KRAS'


In [ ]:
target_id, df = ot.has_target_moa(gene_or_ensembl=symbol)

print(len(df), f'moa found for target {symbol} {target_id}.' )
df.head(3)

In [ ]:
target_id, df = ot.get_reactome_pathways_for_target(symbol)

print(len(df), f'reactome evidences found for target {symbol} {target_id}.' )
df

In [ ]:
target_id, df = ot.get_reactome_disease_evidence_for_target(symbol)

print(len(df), f'reactome disease evidences found for target {symbol} {target_id}.' )
df


In [ ]:
df = ot.con.execute(
    """
    SELECT
        t.id AS targetId,
        t.approvedSymbol,
        t.approvedName,

        p.pathwayId AS pathway_id,
        p.pathway AS pathway,
        p.topLevelTerm AS top_level_term

    FROM target t
    LEFT JOIN UNNEST(t.pathways) AS x(p) ON TRUE

    WHERE t.id = ?

    ORDER BY
        p.topLevelTerm NULLS LAST,
        p.pathway NULLS LAST
    """,
    [target_id],
).df()

df

### Find a gene

In [ ]:
print(ot.con.execute("SHOW TABLES").df())

In [ ]:
for table in ot.tables.keys():
    print("\n###", table, end=" ")
    print(ot.con.execute(f"SELECT COUNT(*) AS n FROM {table}").df())

In [ ]:
ot.show_schema_summary()

### Then test target-specific retrieval with TP53:

In [ ]:
symbol = 'TP53'
symbol = 'PCYT2'
symbol = 'CCNB2'
symbol = 'BLM'

# Em adenocarcinoma pancreático, o gene KRAS mutado ativa uma proteína chamada KLF5, que coordena simultaneamente a desregulação de quatro enzimas da via de síntese de membranas celulares — 
# criando um perfil metabólico identificável por biópsia com implicações prognósticas e terapêuticas.
symbol = 'KLF5'
symbol = 'LPCAT1'
symbol = 'PTGS2'
symbol = 'COX2'
symbol = 'KRAS'
symbol = 'CD274'
symbol = 'PTEN'

dft = ot.resolve_target(symbol)

if dft.empty:
    print("Nothing was found")
    target_id = ''
else:
    target_id = dft.iloc[0]["target_id"]

target_id

### Test Cancer Gene Census

In [ ]:
df = ot.is_target_related_to_cancer(target_id)

print(">>> target_id", target_id)
print(len(df), 'diseases found.')
df.head(2).T

In [ ]:
df = ot.is_target_related_to_disease(target_id=target_id, disease='melanoma')

print(len(df), 'diseases found.')
df.head(3)

In [ ]:
ot.schema("drug_moa")

In [ ]:
df = ot.con.execute(
    """
    SELECT
        kd.drugId,
        kd.prefName,
        kd.targetId,
        moa.chemblIds,
        moa.targets,
        moa.mechanismOfAction,
        moa.actionType
    FROM known_drug kd
    LEFT JOIN drug_moa moa
        ON list_contains(moa.chemblIds, kd.drugId)
       AND list_contains(moa.targets, kd.targetId)
    WHERE kd.drugId IS NOT NULL
    LIMIT 20
    """
).df()

df.head(3)


In [ ]:
disease='melanoma'

df = ot.get_drugs_for_disease(disease=disease, limit=None)

print(len(df), f'drugs found for {disease}.')
df.head(2).T


In [ ]:
df = ot.get_drugs_for_disease_with_moa(disease=disease, limit=None)

print(len(df), f'drugs found for {disease}.')
df.head(2).T

### Test Reactome evidence

In [ ]:
ot.schema("reactome")

In [ ]:
symbol = 'LPCAT1'
symbol = 'PTGS2'
symbol = 'COX2'
symbol = 'KRAS'
symbol = 'CD274'
symbol = 'KLF5'
symbol = 'TP53'

dft = ot.resolve_target(symbol)

if dft.empty:
    print("Nothing was found")
    target_id = ''
else:
    target_id = dft.iloc[0]["target_id"]

print(">>>", target_id)

In [ ]:
target_id, df = ot.get_reactome_pathways_for_target(gene_or_ensembl=symbol)

print(len(df), f'reactome evidences found for target {symbol} {target_id}.' )
df.head(3)

In [ ]:
target_id, df = ot.get_reactome_disease_evidence_for_target(gene_or_ensembl=symbol)

print(len(df), f'reactome disease evidences found for target {symbol} {target_id}.' )
df.head(3)

### drug mechanism of action

In [ ]:
target_id, df = ot.has_target_moa(gene_or_ensembl=symbol)

print(len(df), f'moa found for target {symbol} {target_id}.' )
df.head(3)

### interactions

In [ ]:
ot.schema("interactions")

In [ ]:
symbolA = "FOS"
symbolB = "JUN"

target_idA, target_idB, df = ot.has_target_interactions(gene_or_ensemblA=symbolA, gene_or_ensemblB=symbolB)

print(len(df), f' interactions between {symbolA} ({target_idA}) and {symbolB} ({target_idB}).' )
df.head(3)